# What makes a movie "good" / Oscar-worthy? — Cleaned analysis pipeline
This notebook:
- Loads Rotten Tomatoes (RT), TMDB, and Oscars datasets
- Cleans dates and titles
- Deduplicates TMDB (by popularity)
- Tags RT/TMDB movies with Oscar nomination/win info (allowing ±1 year)
- Performs a robust two-stage merge (title → original title)
- Exports cleaned merged dataset and diagnostics


In [75]:
# Setup & imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import unicodedata
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [76]:
# Load raw datasets
rt_reviews = pd.read_csv('rotten_tomatoes_critic_reviews.csv')
rt_movies = pd.read_csv('rotten_tomatoes_movies.csv')
oscars = pd.read_csv('the_oscar_award.csv')
tmdb = pd.read_csv('TMDB_movie_dataset_v11.csv')

print("Initial data shapes:")
print(f"RT Reviews: {rt_reviews.shape}")
print(f"RT Movies: {rt_movies.shape}")
print(f"Oscars: {oscars.shape}")
print(f"TMDB: {tmdb.shape}")

Initial data shapes:
RT Reviews: (1130017, 8)
RT Movies: (17712, 22)
Oscars: (11110, 8)
TMDB: (1320755, 24)


In [77]:
# Parse dates and compute common cutoff year
rt_movies['release_year'] = pd.to_datetime(rt_movies.get('original_release_date', None),
                                           errors='coerce').dt.year.astype('Int64')
tmdb['release_year'] = pd.to_datetime(tmdb.get('release_date', None),
                                     errors='coerce').dt.year.astype('Int64')

# Oscars dataset likely has 'year_film' column but ensure exists
if 'year_film' not in oscars.columns and 'year' in oscars.columns:
    oscars = oscars.rename(columns={'year': 'year_film'})

max_rt = int(rt_movies['release_year'].dropna().max()) if rt_movies['release_year'].notna().any() else None
max_tmdb = int(tmdb['release_year'].dropna().max()) if tmdb['release_year'].notna().any() else None
max_oscars = int(oscars['year_film'].dropna().max()) if 'year_film' in oscars.columns else None

# Compute cutoff as the minimum of available maxima (guard None)
cutoff = min([x for x in [max_rt, max_tmdb, max_oscars] if x is not None])
print(f"Cutoff year chosen: {cutoff}")

Cutoff year chosen: 2020


In [78]:
# Basic filtering & deduplication
rt_movies = rt_movies[rt_movies['release_year'].notna() & (rt_movies['release_year'] <= cutoff)].copy()
tmdb = tmdb[tmdb['release_year'].notna() & (tmdb['release_year'] <= cutoff)].copy()
oscars = oscars[oscars['year_film'].notna() & (oscars['year_film'] <= cutoff)].copy()

# Drop obvious duplicates
rt_reviews = rt_reviews.drop_duplicates()
rt_movies = rt_movies.drop_duplicates(subset=['rotten_tomatoes_link'])
tmdb = tmdb.drop_duplicates(subset=['id'])
oscars = oscars.drop_duplicates()

print("Post-filter shapes:")
print(f"RT Movies: {rt_movies.shape}")
print(f"TMDB: {tmdb.shape}")
print(f"Oscars: {oscars.shape}")

Post-filter shapes:
RT Movies: (16546, 23)
TMDB: (841151, 25)
Oscars: (10606, 8)


In [79]:
# Title cleaning functions (gentle, preserves numbers and colons; removes accents)
def strip_accents(s):
    if pd.isna(s):
        return ''
    s = str(s)
    nkfd_form = unicodedata.normalize('NFKD', s)
    return nkfd_form.encode('ASCII', 'ignore').decode('ASCII')

def clean_title_gentle(title):
    if pd.isna(title):
        return ''
    t = strip_accents(title.lower())
    # Keep alphanum, spaces, colon, ampersand, hyphen (often meaningful)
    t = re.sub(r"[^a-z0-9:&\-\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# apply
rt_movies['clean_title'] = rt_movies['movie_title'].apply(clean_title_gentle)
tmdb['clean_title'] = tmdb['title'].apply(clean_title_gentle)
# original_title might be NaN for some TMDB rows
tmdb['clean_original_title'] = tmdb.get('original_title', tmdb['title']).apply(clean_title_gentle)

# Quick checks
print("Example cleaned titles (RT):")
print(rt_movies['clean_title'].dropna().head(5).tolist())
print("Example cleaned titles (TMDB):")
print(tmdb[['title', 'clean_title', 'original_title', 'clean_original_title']].head(5))

Example cleaned titles (RT):
['percy jackson & the olympians: the lightning thief', 'please give', '10', '12 angry men twelve angry men', '20000 leagues under the sea']
Example cleaned titles (TMDB):
             title      clean_title   original_title clean_original_title
0        Inception        inception        Inception            inception
1     Interstellar     interstellar     Interstellar         interstellar
2  The Dark Knight  the dark knight  The Dark Knight      the dark knight
3           Avatar           avatar           Avatar               avatar
4     The Avengers     the avengers     The Avengers         the avengers


In [80]:
# Prepare Oscars aggregated table (was_nominated / won_oscar)
# Normalize film names for matching using same cleaning function
oscars['film_clean'] = oscars['film'].astype(str).apply(clean_title_gentle)
oscars_group = (
    oscars
    .assign(won_flag = oscars['winner'].astype(bool))
    .groupby(['film_clean', 'year_film'], as_index=False)
    .agg(won_oscar=('won_flag', 'any'))
)
oscars_group['was_nominated'] = True

print("Oscars grouped sample:")
display(oscars_group.head())

Oscars grouped sample:


,film_clean,year_film,won_oscar,was_nominated
0,10,1979,False,True
1,1000 a minute,1935,False,True
2,102 dalmatians,2000,False,True
3,12,2007,False,True
4,12 angry men,1957,False,True


In [81]:
# Filter TMDB for likely film records and dedupe
# Heuristics: remove adult, require some vote_count or runtime or revenue info to avoid tv entries
tmdb_filtered = tmdb.copy()

# drop rows that look obviously not standard movies
if 'adult' in tmdb_filtered.columns:
    tmdb_filtered = tmdb_filtered[tmdb_filtered['adult'].astype(bool) == False]

# Keep rows with at least a title and non-null id
tmdb_filtered = tmdb_filtered[tmdb_filtered['title'].notna() & tmdb_filtered['id'].notna()]

# Deduplicate: keep one TMDB row per (clean_title, release_year) by vote_count (popularity)
# If vote_count missing, treat as 0
tmdb_filtered['vote_count'] = tmdb_filtered['vote_count'].fillna(0)
tmdb_unique = (
    tmdb_filtered
    .sort_values('vote_count', ascending=False)
    .drop_duplicates(subset=['clean_title', 'release_year'], keep='first')
    .copy()
)

# Also build a deduped original-title table for the second-stage match (dedupe by vote_count)
tmdb_by_original = (
    tmdb_filtered
    .sort_values('vote_count', ascending=False)
    .drop_duplicates(subset=['clean_original_title', 'release_year'], keep='first')
    .copy()
)

print(f"TMDB filtered: {tmdb_filtered.shape}, TMDB unique (by title-year): {tmdb_unique.shape}")
print(f"TMDB unique (by original-title-year): {tmdb_by_original.shape}")

TMDB filtered: (751664, 27), TMDB unique (by title-year): (723526, 27)
TMDB unique (by original-title-year): (636393, 27)


In [82]:
# Tag with Oscars allowing ±1 year
# We'll create a helper to merge with +/-1 year tolerance by expanding oscars_group with -1,0,1 offsets
oscars_tolerant = pd.concat([
    oscars_group.assign(match_year = oscars_group['year_film'] + delta)
    for delta in (-1, 0, 1)
], ignore_index=True)

oscars_tolerant = oscars_tolerant.rename(columns={'year_film': 'oscars_year'})

# Tag RT movies (merge on clean title and tolerant year)
rt_movies = rt_movies.merge(
    oscars_tolerant[['film_clean', 'match_year', 'was_nominated', 'won_oscar']],
    left_on=['clean_title', 'release_year'],
    right_on=['film_clean', 'match_year'],
    how='left'
)
rt_movies['was_nominated'] = rt_movies['was_nominated'].fillna(False)
rt_movies['won_oscar'] = rt_movies['won_oscar'].fillna(False)
rt_movies = rt_movies.drop(columns=['film_clean', 'match_year'], errors='ignore')

# Tag TMDB unique similarly
tmdb_unique = tmdb_unique.merge(
    oscars_tolerant[['film_clean', 'match_year', 'was_nominated', 'won_oscar']],
    left_on=['clean_title', 'release_year'],
    right_on=['film_clean', 'match_year'],
    how='left'
)
tmdb_unique['was_nominated'] = tmdb_unique['was_nominated'].fillna(False)
tmdb_unique['won_oscar'] = tmdb_unique['won_oscar'].fillna(False)
tmdb_unique = tmdb_unique.drop(columns=['film_clean', 'match_year'], errors='ignore')

# Also tag tmdb_by_original for the second stage (original title matches)
tmdb_by_original = tmdb_by_original.merge(
    oscars_tolerant[['film_clean', 'match_year', 'was_nominated', 'won_oscar']],
    left_on=['clean_original_title', 'release_year'],
    right_on=['film_clean', 'match_year'],
    how='left'
)
tmdb_by_original['was_nominated'] = tmdb_by_original['was_nominated'].fillna(False)
tmdb_by_original['won_oscar'] = tmdb_by_original['won_oscar'].fillna(False)
tmdb_by_original = tmdb_by_original.drop(columns=['film_clean', 'match_year'], errors='ignore')

print("Oscar tagging complete.")

Oscar tagging complete.


In [83]:
# Two-stage merge with diagnostics
# Stage 1: inner join on clean_title & release_year using tmdb_unique
stage1 = rt_movies.merge(
    tmdb_unique.add_suffix('_tmdb'),
    left_on=['clean_title', 'release_year'],
    right_on=['clean_title_tmdb', 'release_year_tmdb'],
    how='inner'
)
print(f"Stage1 matches: {len(stage1)}")

# Identify RT rows not matched in stage1
rt_unmatched = rt_movies.loc[~rt_movies.index.isin(stage1['index'] if 'index' in stage1.columns else [])].copy()
# Simpler way to get unmatched: use a left merge indicator
left_check = rt_movies.merge(
    tmdb_unique[['clean_title', 'release_year', 'id']].rename(columns={'id':'tmdb_id'}),
    on=['clean_title', 'release_year'],
    how='left',
    indicator=True
)
rt_unmatched = rt_movies[left_check['_merge'] == 'left_only'].copy()
print(f"RT unmatched after stage1: {len(rt_unmatched)}")

# Stage 2: match remaining RT unmatched to tmdb_by_original on (clean_title -> clean_original_title) & year
stage2 = rt_unmatched.merge(
    tmdb_by_original.add_suffix('_tmdb'),
    left_on=['clean_title', 'release_year'],
    right_on=['clean_original_title_tmdb', 'release_year_tmdb'],
    how='inner'
)
print(f"Stage2 matches: {len(stage2)}")

# Combine matched results
matched = pd.concat([stage1, stage2], ignore_index=True, sort=False)

# Basic diagnostics
num_rt = len(rt_movies)
num_tmdb_unique = len(tmdb_unique)
num_matched = len(matched)
print(f"RT count: {num_rt}, TMDB unique count: {num_tmdb_unique}, Matched: {num_matched}")
print(f"Match rate (RT matched / RT total): {num_matched/num_rt*100:.2f}%")

Stage1 matches: 11986
RT unmatched after stage1: 4577
Stage2 matches: 3022
RT count: 16553, TMDB unique count: 723544, Matched: 15008
Match rate (RT matched / RT total): 90.67%


In [84]:
# Many-to-many safety & collapse duplicates if any
# If there are duplicated RT entries in matched (i.e., same RT link matched to multiple TMDB ids),
# keep the TMDB record with highest tmdb vote_count_tmdb (if available), otherwise keep first.

if 'tmdb_unique_vote_count_tmdb' in matched.columns:
    # column may vary; attempt to find vote_count column name
    vote_cols = [c for c in matched.columns if 'vote_count' in c]
else:
    vote_cols = [c for c in matched.columns if 'vote_count' in c]

print("vote_count cols found in matched:", vote_cols)

# define a function to collapse duplicates by choosing max vote_count if present
def collapse_matches(df):
    # Expect column naming like 'rotten_tomatoes_link', and 'id_tmdb' for tmdb id
    if 'rotten_tomatoes_link' in df.columns and any('id_tmdb' in c for c in df.columns):
        key = 'rotten_tomatoes_link'
        # choose row with max vote_count_tmdb if exists
        vote_col = [c for c in df.columns if c.endswith('vote_count_tmdb')]
        if vote_col:
            return df.sort_values(vote_col[0], ascending=False).drop_duplicates(subset=[key], keep='first')
        else:
            return df.drop_duplicates(subset=[key], keep='first')
    else:
        return df.drop_duplicates()

matched_collapsed = collapse_matches(matched)

print(f"Matched collapsed from {len(matched)} to {len(matched_collapsed)}")

vote_count cols found in matched: ['vote_count_tmdb']
Matched collapsed from 15008 to 11978


In [85]:
# Final assembled dataset and save
# We'll create a tidy merged df combining key RT and TMDB columns

# Identify columns from RT and TMDB suffixes
rt_cols = [c for c in matched_collapsed.columns if not c.endswith('_tmdb')]
tmdb_cols = [c for c in matched_collapsed.columns if c.endswith('_tmdb')]

# Normalize tmdb column names by stripping suffix
tmdb_renames = {c: c.replace('_tmdb', '') for c in tmdb_cols}

final = matched_collapsed.copy()
final = final.rename(columns=tmdb_renames)

# Keep a recommended subset of columns (extend as needed)
keep_cols = keep_cols = [
    'movie_title', 'release_year',
    'content_rating', 'genres', 'directors', 'actors',
    'runtime', 'production_company', 'production_countries',
    'tomatometer_rating', 'audience_rating',
    'was_nominated', 'won_oscar',
    'vote_average', 'vote_count',
    'revenue', 'budget', 'keywords'
]

keep_cols_available = [c for c in keep_cols if c in final.columns]

# Remove duplicate columns, keep first occurrence
final_movies = final_movies.loc[:, ~final_movies.columns.duplicated()]

final_movies = final[keep_cols_available].copy()

In [87]:
final_movies = final_movies.loc[:, ~final_movies.columns.duplicated()]
final_movies = final_movies.drop(columns=['vote_count'])

In [90]:
final_movies.to_csv('merged_movies_rt_tmdb.csv', index=False)
print(f"Final movies saved with {len(final_movies)} rows.")

Final movies saved with 11978 rows.


In [99]:
def split_column(x):
    if pd.isna(x) or x.strip() == "":
        return []
    # Split by comma, strip whitespace
    return [item.strip() for item in x.split(",") if item.strip()]

list_columns = ['genres', 'directors', 'actors', 'production_countries', 'keywords']

for col in list_columns:
    if col in final_movies.columns:
        final_movies[col] = final_movies[col].apply(split_column)

In [100]:
# Ensure unique movie ID
if 'movie_id' not in final_movies.columns:
    final_movies = final_movies.reset_index().rename(columns={'index': 'movie_id'})

for col in list_columns:
    exploded = final_movies[['movie_id', col]].explode(col).dropna(subset=[col])
    exploded = exploded.rename(columns={col: col[:-1] if col.endswith('s') else col})  # e.g., genres -> genre
    exploded.to_csv(f'movie_{col}.csv', index=False)
    print(f"Saved exploded file for {col}, rows: {len(exploded)}")

Saved exploded file for genres, rows: 26591
Saved exploded file for directors, rows: 13429
Saved exploded file for actors, rows: 335211
Saved exploded file for production_countries, rows: 15319
Saved exploded file for keywords, rows: 94085
